# Checking how to enable PyApprox functionality in SPAROW

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pyapprox.util.backends.numpy import NumpyBkd
from pyapprox_benchmarks.statest import (
    PolynomialEnsembleBenchmark,
)
from pyapprox.statest.statistics import MultiOutputMean
from pyapprox.statest.mc_estimator import MCEstimator
from pyapprox.statest import (
    MLMCEstimator, MFMCEstimator, GMFEstimator, GRDEstimator, GISEstimator,
)
from pyapprox.statest.acv import ACVAllocator, default_allocator_factory
from pyapprox.statest.acv.base import FittedACVEstimator
from pyapprox.statest.acv.search import ACVSearch
from pyapprox.statest.acv.strategies import (
    FixedRecursionStrategy, TreeDepthRecursionStrategy,
)
from pyapprox.statest.allocation import MCAllocator, CVAllocator
from pyapprox.statest.plotting import (
    plot_allocation, plot_estimator_variance_reductions,
)
from pyapprox.optimization.minimize.scipy.slsqp import ScipySLSQPOptimizer

bkd = NumpyBkd()
np.random.seed(42)

# SLSQP is more robust than the default trust-constr optimizer for ACV allocation
optimizer = ScipySLSQPOptimizer(maxiter=200)
allocator_factory = lambda est: default_allocator_factory(est, optimizer=optimizer)

benchmark = PolynomialEnsembleBenchmark(bkd, nmodels=5)
models    = benchmark.problem().models()        # f_0 (HF) ... f_4 (cheapest)
variable  = benchmark.problem().prior()
costs     = benchmark.problem().costs()
nqoi      = models[0].nqoi()
nmodels   = len(models)

costs_np = bkd.to_numpy(costs)
print(f"{nmodels} models, {nqoi} QoI(s)")
for a, c in enumerate(costs_np):
    print(f"  model {a}: cost = {c:.5f}")

5 models, 1 QoI(s)
  model 0: cost = 1.00000
  model 1: cost = 0.10000
  model 2: cost = 0.01000
  model 3: cost = 0.00100
  model 4: cost = 0.00010


## Stage 1: Pilot Study

In [2]:
N_pilot = 50
samples_pilot = variable.rvs(N_pilot)
vals_pilot = [m(samples_pilot) for m in models]

stat = MultiOutputMean(nqoi, bkd)
cov_pilot, = stat.compute_pilot_quantities(vals_pilot)
stat.set_pilot_quantities(cov_pilot)

# Inspect pilot correlations with HF model
cov_np = bkd.to_numpy(cov_pilot)
for a in range(1, nmodels):
    rho = cov_np[0, a] / np.sqrt(cov_np[0, 0] * cov_np[a, a])
    print(f"  ρ(f0, f{a}) = {rho:.4f}")

  ρ(f0, f1) = 0.9954
  ρ(f0, f2) = 0.9756
  ρ(f0, f3) = 0.9246
  ρ(f0, f4) = 0.8112


In [3]:
total_budget = 500.0
pilot_cost   = float(costs_np.sum()) * N_pilot
remaining    = total_budget - pilot_cost
print(f"Pilot cost: {pilot_cost:.1f}  |  Remaining: {remaining:.1f}")

Pilot cost: 55.6  |  Remaining: 444.4


## Stage 2: Build Estimator and Allocate

In [4]:
est = MFMCEstimator(stat, costs)
allocator = default_allocator_factory(est)
result = allocator.allocate(remaining)
fitted = FittedACVEstimator(est, result)

print(f"Samples per model: {fitted.nsamples_per_model()}")
print(f"Predicted std:     {float(fitted.covariance()[0,0])**0.5:.6f}")

Samples per model: [   200   1313   6548  29506 170626]
Predicted std:     0.002432


## Stage 3: Generate Samples

In [5]:
samples_per_model = fitted.generate_samples_per_model(variable.rvs)
print(f"Sample shapes: {[s.shape for s in samples_per_model]}")

Sample shapes: [(1, 200), (1, 1313), (1, 6548), (1, 29506), (1, 170626)]


## Stage 4: Evaluate Models

In [6]:
values_per_model = [models[a](samples_per_model[a]) for a in range(nmodels)]

## Stage 5: Compute Estimate

In [7]:
estimate = fitted(values_per_model)

true_mean = float(bkd.to_numpy(benchmark.ensemble_means()[0, 0]))
print(f"Estimate:  {float(estimate):.6f}")
print(f"True mean: {true_mean:.6f}")
print(f"Error:     {abs(float(estimate) - true_mean):.6f}")

Estimate:  0.167531
True mean: 0.166667
Error:     0.000864


/tmp/ipykernel_603227/1696033056.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"Estimate:  {float(estimate):.6f}")
/tmp/ipykernel_603227/1696033056.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"Error:     {abs(float(estimate) - true_mean):.6f}")


## Compare againt MC

In [8]:
stat_mc = MultiOutputMean(nqoi, bkd)
stat_mc.set_pilot_quantities(cov_pilot[:1, :1])
mc_est = MCEstimator(stat_mc, costs[:1])
mc_fitted = MCAllocator(mc_est).allocate(remaining)

mc_var  = float(mc_fitted.covariance()[0, 0])
mf_var  = float(fitted.covariance()[0, 0])
print(f"MC std:  {mc_var**0.5:.6f}")
print(f"MF std:  {mf_var**0.5:.6f}")
print(f"Variance reduction: {mc_var / mf_var:.1f}×")

MC std:  0.011500
MF std:  0.002432
Variance reduction: 22.4×
